# KL vs VQ First-Stage Autoencoder Experiment

**Minimal Kaggle Launcher**

This notebook orchestrates training and evaluation. Implementation is in `src/` and `scripts/`.

## Setup Requirements
1. **GPU**: Enable T4 GPU in Kaggle settings
2. **Dataset**: Add No_Mixed thermal dataset to notebook
3. **Repository**: Update GitHub URL below

## Expected Runtime
- KL Training: ~1-2 hours (100 epochs, early stopping)
- VQ Training: ~1-2 hours (100 epochs, early stopping)
- Evaluation: ~15 minutes
- **Total**: ~3-4 hours

In [ ]:
# Cell 1: Environment Setup
import os
import sys
from pathlib import Path
import shutil

# Detect environment
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_INPUT.exists()

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Current directory: {os.getcwd()}")

# Clone repository
if IS_KAGGLE:
    REPO_NAME = 'Mixed-Pixels'
    GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # UPDATE THIS!
    
    if not (KAGGLE_WORKING / REPO_NAME).exists():
        print(f"\nCloning repository from {GITHUB_URL}...")
        !git clone {GITHUB_URL} /kaggle/working/{REPO_NAME}
    else:
        print(f"\nRepository already exists, pulling latest...")
        !cd /kaggle/working/{REPO_NAME} && git pull
    
    os.chdir(KAGGLE_WORKING / REPO_NAME)
else:
    # Local: navigate to repo root
    repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    os.chdir(repo_root)

print(f"Working directory: {os.getcwd()}")
sys.path.insert(0, str(Path.cwd() / 'src'))

# Verify GPU
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected! Training will be very slow.")

In [ ]:
# Cell 2: Configure Dataset Paths for Kaggle
import yaml

# Determine which config to use based on available hardware
if IS_KAGGLE:
    # Check GPU count
    import torch
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    
    if num_gpus >= 2:
        print(f"✨ Detected {num_gpus} GPUs - using optimized 2xGPU configs")
        kl_config = 'configs/experiment1/kl_autoencoder_kaggle_2gpu.yaml'
        vq_config = 'configs/experiment1/vq_autoencoder_kaggle_2gpu.yaml'
    else:
        print(f"Using single GPU config")
        kl_config = 'configs/experiment1/kl_autoencoder.yaml'
        vq_config = 'configs/experiment1/vq_autoencoder.yaml'
    
    # Update dataset path
    KAGGLE_DATASET_PATH = '/kaggle/input/no-mixed-thermal'  # UPDATE IF DIFFERENT
    
    print(f"Configuring dataset path: {KAGGLE_DATASET_PATH}")
    
    for config_file in [kl_config, vq_config]:
        with open(config_file, 'r') as f:
            config = yaml.safe_load(f)
        
        # Update dataset path
        config['data']['dataset_root'] = KAGGLE_DATASET_PATH
        
        with open(config_file, 'w') as f:
            yaml.dump(config, f, default_flow_style=False)
        
        print(f"  ✓ Updated {config_file}")
    
    print("\nVerifying dataset...")
    dataset_path = Path(KAGGLE_DATASET_PATH)
    if dataset_path.exists():
        num_images = len(list(dataset_path.rglob('*.png'))) + len(list(dataset_path.rglob('*.jpg')))
        print(f"  ✓ Dataset found: {num_images} images")
    else:
        print(f"  ✗ Dataset not found at {KAGGLE_DATASET_PATH}")
        print("  Available datasets:")
        for item in KAGGLE_INPUT.iterdir():
            print(f"    - {item.name}")
    
    # Store config paths for later cells
    KL_CONFIG = kl_config
    VQ_CONFIG = vq_config
else:
    print("Running locally - using paths from configs")
    KL_CONFIG = 'configs/experiment1/kl_autoencoder.yaml'
    VQ_CONFIG = 'configs/experiment1/vq_autoencoder.yaml'

In [ ]:
# Cell 3: Train KL Autoencoder
print("="*70)
print("TRAINING KL AUTOENCODER")
print("="*70)

!python scripts/training/train_first_stage.py configs/experiment1/kl_autoencoder.yaml

print("\n✓ KL training complete")

In [ ]:
# Cell 4: Train VQ Autoencoder
print("="*70)
print("TRAINING VQ AUTOENCODER")
print("="*70)

!python scripts/training/train_first_stage.py configs/experiment1/vq_autoencoder.yaml

print("\n✓ VQ training complete")

In [ ]:
# Cell 5: Evaluate and Save Results
print("="*70)
print("EVALUATION")
print("="*70)

# Evaluate KL
print("\nEvaluating KL autoencoder...")
!python scripts/evaluation/evaluate_first_stage.py \
    --checkpoint checkpoints/experiment1/kl_no_mixed/best_model.pth \
    --config configs/experiment1/kl_autoencoder.yaml \
    --output outputs/experiment1/kl_evaluation/

# Evaluate VQ
print("\nEvaluating VQ autoencoder...")
!python scripts/evaluation/evaluate_first_stage.py \
    --checkpoint checkpoints/experiment1/vq_no_mixed/best_model.pth \
    --config configs/experiment1/vq_autoencoder.yaml \
    --output outputs/experiment1/vq_evaluation/

# Compare results
print("\nComparing results...")
!python scripts/analysis/compare_kl_vq.py \
    --kl-results outputs/experiment1/kl_evaluation/ \
    --vq-results outputs/experiment1/vq_evaluation/ \
    --output outputs/experiment1/comparison/

# Save results for download
if IS_KAGGLE:
    print("\nSaving results to /kaggle/working for download...")
    
    # Copy checkpoints (best models only to save space)
    output_dir = Path('/kaggle/working/experiment1_results')
    output_dir.mkdir(exist_ok=True)
    
    shutil.copy('checkpoints/experiment1/kl_no_mixed/best_model.pth', 
                output_dir / 'kl_best_model.pth')
    shutil.copy('checkpoints/experiment1/vq_no_mixed/best_model.pth', 
                output_dir / 'vq_best_model.pth')
    
    # Copy evaluation results
    shutil.copytree('outputs/experiment1', output_dir / 'outputs', dirs_exist_ok=True)
    
    # Copy training logs
    shutil.copytree('logs/experiment1', output_dir / 'logs', dirs_exist_ok=True)
    
    print(f"✓ Results saved to {output_dir}")
    print("\nDownload 'experiment1_results' folder from Kaggle output")

print("\n" + "="*70)
print("EXPERIMENT COMPLETE!")
print("="*70)